# Does the rural PHC staffing–IMR association survive socioeconomic controls?

This exploratory state/UT cross-section relates the 2023 rural PHC doctor staffing ratio to 2023 infant mortality per 1,000 live births, controlling for (1) log per-capita NSDP at current prices in 2021–22 and (2) NFHS-5 female literacy among women aged 15–49 in 2019–21. This is an observational, small-sample association—not a causal design.

**Main specification:** use the natural logarithm of the staffing ratio. The raw ratio is heavily right-skewed: a small number of states/UTs have very high in-position-to-required ratios, giving the upper tail substantial leverage in a linear specification. Logging compresses that tail and models proportional differences in the staffing ratio. The original unlogged ratio remains as a sensitivity specification so the transformation's effect is transparent.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

DATA = Path('../Dataset')
staff = pd.read_csv(DATA / 'state_rural_phc_doctors_2023.csv')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
socio = pd.read_csv(DATA / 'state_socioeconomic_covariates_nfhs5_rbi.csv')
print('Staff rows:', len(staff), '| IMR rows:', len(imr), '| socioeconomic rows:', len(socio))
socio.head()

Staff rows: 36 | IMR rows: 37 | socioeconomic rows: 36


,state_ut,female_literacy_nfhs5_pct,nfhs_reference_period,per_capita_nsdp_2021_22_inr,nsdp_reference_year
0,Andaman & Nicobar Islands,80.5,2019-21,229570.0,2021-22
1,Andhra Pradesh,66.7,2019-21,197214.0,2021-22
2,Arunachal Pradesh,71.3,2019-21,190851.0,2021-22
3,Assam,75.1,2019-21,103371.0,2021-22
4,Bihar,55.0,2019-21,47296.0,2021-22


## 1. Build a transparent complete-case sample

The staffing measure is in-position rural PHC doctors divided by the source-reported required rural PHC doctors. It is not doctors per capita. The outcome is 2023 infant mortality per 1,000 live births. The model controls for log nominal per-capita NSDP (₹, 2021–22) and NFHS-5 female literacy (2019–21). The combined Dadra & Nagar Haveli and Daman & Diu UT and Lakshadweep lack a separate 2021–22 NSDP observation in the RBI table; these two matched observations are excluded from the adjusted complete-case sample rather than imputed. The workforce–IMR match itself does not include Ladakh.

In [2]:
imr = imr.loc[imr['state_ut'] != 'All India'].copy()
imr['state_key'] = imr['state_ut'].replace({'Dadra & Nagar Haveli (includes Daman & Diu)': 'Dadra & Nagar Haveli & Daman & Diu'})
staff['state_key'] = staff['state_ut']
base = staff.merge(imr[['state_key','infant_mortality_rate_2023_per_1000_live_births']], on='state_key', how='inner', validate='one_to_one')
base['staffing_ratio'] = base['in_position_rural_phc_doctors'] / base['required_rural_phc_doctors'].where(base['required_rural_phc_doctors'] > 0)
base = base.dropna(subset=['staffing_ratio','infant_mortality_rate_2023_per_1000_live_births']).copy()
analysis = base.merge(socio, left_on='state_ut', right_on='state_ut', how='left', validate='one_to_one')
analysis['log_nsdp'] = np.log(pd.to_numeric(analysis['per_capita_nsdp_2021_22_inr'], errors='coerce'))
analysis = analysis.dropna(subset=['female_literacy_nfhs5_pct','log_nsdp']).copy()
print('Bivariate matched sample:', len(base))
print('Adjusted complete-case sample:', len(analysis))
print('Excluded from adjusted model due to missing NSDP:', sorted(set(base.state_ut) - set(analysis.state_ut)))
print('Reference years: IMR/staffing=2023; female literacy=2019–21; NSDP=2021–22')
analysis[['state_ut','staffing_ratio','infant_mortality_rate_2023_per_1000_live_births','per_capita_nsdp_2021_22_inr','female_literacy_nfhs5_pct']].sort_values('state_ut').head(10)

Bivariate matched sample: 34
Adjusted complete-case sample: 32
Excluded from adjusted model due to missing NSDP: ['Dadra & Nagar Haveli & Daman & Diu', 'Lakshadweep']
Reference years: IMR/staffing=2023; female literacy=2019–21; NSDP=2021–22


,state_ut,staffing_ratio,infant_mortality_rate_2023_per_1000_live_births,per_capita_nsdp_2021_22_inr,female_literacy_nfhs5_pct
28,Andaman & Nicobar Islands,2.272727,9.0,229570.0,80.5
0,Andhra Pradesh,2.002620,19.0,197214.0,66.7
1,Arunachal Pradesh,1.070866,20.0,190851.0,71.3
2,Assam,1.392391,30.0,103371.0,75.1
3,Bihar,1.938776,23.0,47296.0,55.0
4,Chhattisgarh,0.756792,37.0,122934.0,72.5
30,Delhi,3.200000,14.0,376217.0,83.7
5,Goa,4.619048,6.0,459094.0,92.2
6,Gujarat,1.050573,20.0,241584.0,73.4
7,Haryana,1.520833,26.0,264729.0,79.7


## 2. Main adjusted model: log staffing ratio

The main specification uses `log(staffing_ratio)`, alongside log nominal per-capita NSDP (₹, 2021–22) and NFHS-5 female literacy (2019–21). The raw staffing ratio is heavily right-skewed, so its logarithm reduces the leverage of unusually high ratios and makes the staffing term reflect proportional rather than absolute changes. The coefficient is the expected change in IMR (deaths per 1,000 live births) associated with a one-unit increase in log staffing ratio, conditional on the two controls.

For comparison, the table reports the original unlogged-ratio model on the same complete-case sample. All models use HC3 heteroskedasticity-consistent standard errors. Predictors are centred within each estimation sample; centring changes intercepts, not slope estimates.

In [ ]:
outcome = 'infant_mortality_rate_2023_per_1000_live_births'
analysis['log_staffing_ratio'] = np.log(analysis['staffing_ratio'])
analysis['log_nsdp_c'] = analysis['log_nsdp'] - analysis['log_nsdp'].mean()
analysis['female_literacy_c'] = analysis['female_literacy_nfhs5_pct'] - analysis['female_literacy_nfhs5_pct'].mean()
analysis['staffing_ratio_c'] = analysis['staffing_ratio'] - analysis['staffing_ratio'].mean()
analysis['log_staffing_ratio_c'] = analysis['log_staffing_ratio'] - analysis['log_staffing_ratio'].mean()

model_unadjusted = smf.ols(f'{outcome} ~ staffing_ratio', data=analysis).fit(cov_type='HC3')
model_adjusted_linear = smf.ols(f'{outcome} ~ staffing_ratio_c + log_nsdp_c + female_literacy_c', data=analysis).fit(cov_type='HC3')
model_adjusted = smf.ols(f'{outcome} ~ log_staffing_ratio_c + log_nsdp_c + female_literacy_c', data=analysis).fit(cov_type='HC3')

def staffing_result(model, term, specification, sample):
    ci = model.conf_int().loc[term]
    return {
        'specification': specification,
        'sample': sample,
        'n': int(model.nobs),
        'staffing_coefficient': model.params[term],
        'HC3_std_error': model.bse[term],
        'p_value_HC3': model.pvalues[term],
        'ci95_low': ci.iloc[0],
        'ci95_high': ci.iloc[1],
    }

main_results = pd.DataFrame([
    staffing_result(model_adjusted_linear, 'staffing_ratio_c', 'Linear staffing ratio (adjusted)', 'Full complete-case sample'),
    staffing_result(model_adjusted, 'log_staffing_ratio_c', 'Log staffing ratio (main adjusted model)', 'Full complete-case sample'),
])
display(main_results.round(4))
print('Main log-ratio model n:', int(model_adjusted.nobs))
print('Main log-ratio model R-squared:', round(model_adjusted.rsquared, 3))
print(model_adjusted.summary())

In [ ]:
labels = ['Adjusted: linear staffing ratio', 'Adjusted: log staffing ratio (main)']
models = [model_adjusted_linear, model_adjusted]
terms = ['staffing_ratio_c', 'log_staffing_ratio_c']
estimates = [m.params[t] for m, t in zip(models, terms)]
lower = [m.conf_int().loc[t].iloc[0] for m, t in zip(models, terms)]
upper = [m.conf_int().loc[t].iloc[1] for m, t in zip(models, terms)]
errors = [[e-l for e,l in zip(estimates, lower)], [u-e for u,e in zip(upper, estimates)]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(estimates, labels, xerr=errors, fmt='o', capsize=4)
ax.axvline(0, linestyle='--', linewidth=1, color='black')
ax.set_xlabel('Staffing coefficient (IMR deaths per 1,000 live births)')
ax.set_title('Adjusted staffing coefficient: linear and log-ratio specifications')
fig.tight_layout()
fig.savefig('../Visualisations/11_adjusted_regression_staffing_coefficient.svg', format='svg', bbox_inches='tight')
plt.show()

## 3. Specification and influence sensitivity

The table below compares the adjusted staffing coefficient across the original linear-ratio setup, the log-ratio main model, and the linear-ratio model after excluding states/UTs with fewer than 50 required rural PHC doctors. The 50-doctor threshold is a sensitivity check, not a formal statistical cutoff.

A leave-one-state/UT-out analysis then re-estimates the **main log-ratio model** after dropping each observation in turn. This reveals whether the headline log-ratio result depends on one particular state/UT. A final joint exclusion of Goa and Delhi is shown as a targeted influence check. All estimates use HC3 standard errors and the same two socioeconomic controls. HC3 does not solve omitted-variable bias or measurement error.

In [ ]:
# Collinearity diagnostic for the main specification
X = analysis[['log_staffing_ratio','log_nsdp','female_literacy_nfhs5_pct']].copy()
X = (X - X.mean()) / X.std(ddof=0)
X.insert(0, 'const', 1.0)
vif = pd.DataFrame({'variable': X.columns, 'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif.round(3))

# Comparable adjusted sensitivity specifications
restricted = analysis.loc[analysis['required_rural_phc_doctors'] >= 50].copy()
restricted['staffing_ratio_c'] = restricted['staffing_ratio'] - restricted['staffing_ratio'].mean()
restricted['log_nsdp_c'] = restricted['log_nsdp'] - restricted['log_nsdp'].mean()
restricted['female_literacy_c'] = restricted['female_literacy_nfhs5_pct'] - restricted['female_literacy_nfhs5_pct'].mean()
model_restricted = smf.ols(f'{outcome} ~ staffing_ratio_c + log_nsdp_c + female_literacy_c', data=restricted).fit(cov_type='HC3')

comparison_rows = [
    staffing_result(model_adjusted_linear, 'staffing_ratio_c', 'As-is: linear ratio, adjusted', 'Full complete-case sample'),
    staffing_result(model_adjusted, 'log_staffing_ratio_c', 'Log ratio, adjusted (main)', 'Full complete-case sample'),
    staffing_result(model_restricted, 'staffing_ratio_c', 'Exclude required doctors < 50', 'Required doctors >= 50'),
]

# Leave-one-state/UT-out influence analysis of the main log-ratio model
loo_rows = []
for state in analysis['state_ut']:
    sample = analysis.loc[analysis['state_ut'] != state].copy()
    for col in ['log_staffing_ratio', 'log_nsdp', 'female_literacy_nfhs5_pct']:
        sample[col + '_c'] = sample[col] - sample[col].mean()
    fit = smf.ols(f'{outcome} ~ log_staffing_ratio_c + log_nsdp_c + female_literacy_nfhs5_pct_c', data=sample).fit(cov_type='HC3')
    row = staffing_result(fit, 'log_staffing_ratio_c', 'Leave-one-out log ratio', f'Exclude {state}')
    row['excluded_state_ut'] = state
    loo_rows.append(row)

loo = pd.DataFrame(loo_rows)
loo_min = loo.loc[loo['staffing_coefficient'].idxmin()]
loo_max = loo.loc[loo['staffing_coefficient'].idxmax()]
comparison_rows.extend([
    {**loo_min.to_dict(), 'specification': 'Leave-one-out range: most negative', 'sample': f"Exclude {loo_min['excluded_state_ut']}"},
    {**loo_max.to_dict(), 'specification': 'Leave-one-out range: least negative', 'sample': f"Exclude {loo_max['excluded_state_ut']}"},
])

# Targeted joint exclusion highlighted in the robustness discussion
without_goa_delhi = analysis.loc[~analysis['state_ut'].isin(['Goa', 'Delhi'])].copy()
for col in ['log_staffing_ratio', 'log_nsdp', 'female_literacy_nfhs5_pct']:
    without_goa_delhi[col + '_c'] = without_goa_delhi[col] - without_goa_delhi[col].mean()
model_no_goa_delhi = smf.ols(f'{outcome} ~ log_staffing_ratio_c + log_nsdp_c + female_literacy_nfhs5_pct_c', data=without_goa_delhi).fit(cov_type='HC3')
comparison_rows.append(staffing_result(model_no_goa_delhi, 'log_staffing_ratio_c', 'Exclude Goa and Delhi (joint check)', 'Full sample less Goa and Delhi'))

comparison = pd.DataFrame(comparison_rows)
display(comparison[['specification','sample','n','staffing_coefficient','HC3_std_error','p_value_HC3','ci95_low','ci95_high']].round(4))

print('Leave-one-out estimates for every state/UT:')
display(loo[['excluded_state_ut','n','staffing_coefficient','p_value_HC3','ci95_low','ci95_high']].sort_values('staffing_coefficient').round(4))
print('Leave-one-out coefficient range:', round(loo['staffing_coefficient'].min(), 4), 'to', round(loo['staffing_coefficient'].max(), 4))

## 4. Interpretation and limitations

- The log-ratio specification is the main model because the raw staffing ratio is heavily right-skewed; logging compresses extreme upper-tail values. The linear-ratio model is retained as a transparent sensitivity check.
- Read the specification comparison and leave-one-out table together: the estimated staffing association changes with functional form and sample definition. The targeted Goa-and-Delhi joint exclusion is a sensitivity check, not proof that either observation is erroneous.
- A log-ratio coefficient is the change in IMR associated with a one-unit increase in the natural log of the staffing ratio, holding the two controls fixed. It is not directly comparable in magnitude to the coefficient on the unlogged ratio because the units differ.
- The income measure is nominal per-capita NSDP at current prices, not real income or household consumption. State price-level differences are not adjusted.
- NFHS-5 literacy is a survey measure for women aged 15–49; it is not the same construct as the Census literacy rate for all women.
- The model does not control for maternal health, nutrition, sanitation, urbanisation, fertility, health-system quality or other confounders. The control variables also predate the 2023 outcome.
- HC3 addresses heteroskedasticity in standard errors; it does not make the model causal or eliminate small-sample sensitivity.
- Treat all estimates as exploratory. Do not claim that staffing changes cause changes in infant mortality.

In [ ]:
# Reproducibility and data-integrity checks
assert not base.empty, 'No bivariate matched observations; check state keys and source coverage'
assert not analysis.empty, 'No complete-case observations remain for the adjusted model'
assert analysis[['staffing_ratio','log_staffing_ratio','log_nsdp','female_literacy_nfhs5_pct',outcome]].notna().all().all()
assert (analysis['staffing_ratio'] > 0).all()
assert (analysis['per_capita_nsdp_2021_22_inr'] > 0).all()
assert model_adjusted.nobs == len(analysis)
assert np.isfinite(model_adjusted.params).all()
assert model_adjusted.condition_number < 100, model_adjusted.condition_number
assert len(loo) == len(analysis), 'Leave-one-out analysis must contain one row per complete-case state/UT'
print('Bivariate matched n:', len(base))
print('Adjusted complete-case n:', len(analysis))
print('Main log-ratio staffing coefficient:', round(model_adjusted.params['log_staffing_ratio_c'], 4))
print('Main log-ratio HC3 p-value:', round(model_adjusted.pvalues['log_staffing_ratio_c'], 4))
print('Leave-one-out coefficient range:', round(loo['staffing_coefficient'].min(), 4), 'to', round(loo['staffing_coefficient'].max(), 4))
print('Centered-model condition number:', round(model_adjusted.condition_number, 2))
print('Notebook 07 checks passed.')